In [12]:
from pathlib import Path
import pandas as pd

# Define the path of your folder containing 60 zips
zip_folder = Path(r"C:\Users\ASUS\Desktop\tennis_data")

# Read proper tables
MatchEventInfo_df = pd.read_parquet(zip_folder / "event_all.parquet")

MatchHomeTeamInfo_df = pd.read_parquet(zip_folder / "home_team_all.parquet")

MatchAwayTeamInfo_df = pd.read_parquet(zip_folder / "away_team_all.parquet")

In [13]:
# From MatchEventInfo, select match_id and winner_code, then drop duplicates
MatchEventInfo_df = MatchEventInfo_df[["match_id","winner_code"]]
MatchEventInfo_df.drop_duplicates(subset="match_id", inplace=True)

# Select desired columns from MatchHomeTeamInfo and MatchHomeTeamInfo
MatchHomeTeamInfo_df = MatchHomeTeamInfo_df[["match_id","name","current_rank","date"]].copy()
MatchHomeTeamInfo_df.rename(columns={"current_rank":"home_rank","name":"home_name"},inplace=True)

MatchAwayTeamInfo_df = MatchAwayTeamInfo_df[["match_id","name","current_rank","date"]].copy()
MatchAwayTeamInfo_df.rename(columns={"current_rank":"away_rank","name":"away_name"},inplace=True)

# Merge 2 tables to get both teams' information in a row
result_df = MatchAwayTeamInfo_df.merge(MatchHomeTeamInfo_df, on=["match_id","date"])

# Merge the result table with MatchEventInfo to get the winner_code
result_df = result_df.merge(MatchEventInfo_df, on="match_id", how = "left")

# Drop date and null values on winner_code, because we do not need them!
result_df.drop(columns="date", inplace=True)
result_df.dropna(subset=["winner_code"], inplace=True)
result_df.drop_duplicates(inplace=True)

In [14]:
# Create 2 tables and rename to enable group by!
# Home info:
home_df = result_df[["home_name", "away_rank", "winner_code"]].copy()

home_df.rename(columns={
    "home_name": "player",
    "away_rank": "opponent_rank"}, inplace=True)

home_df["won"] = (home_df["winner_code"] == 1).astype(int)

# Away info:
away_df = result_df[["away_name", "home_rank", "winner_code"]].copy()

away_df.rename(columns={
    "away_name": "player",
    "home_rank":"opponent_rank"}, inplace=True)

away_df["won"] = (away_df["winner_code"] == 2).astype(int)

# Now, time to concat and filter players with opponent
players_df = pd.concat([home_df, away_df], ignore_index=True)

players_df = players_df[players_df["opponent_rank"]<=10]

In [28]:
# Final step: get number of wins and total games vs opponents, ranked below 10 to calculate success rate!

stats = (players_df.groupby("player").agg(
        total_matches=("won", "count"),
        wins=("won", "sum")
    )
)

stats["win_percentage"] = (
    stats["wins"] /
    stats["total_matches"] * 100
)

stats = stats.sort_values("win_percentage", ascending=False)
stats.shape

(132, 3)

In [26]:
# Show all stats
pd.set_option("display.max_rows", None)
print(stats)

                      total_matches  wins  win_percentage
player                                                   
Altmaier D.                       2     2      100.000000
Mensik J.                         1     1      100.000000
Bublik A.                         2     2      100.000000
Avanesyan E.                      1     1      100.000000
Świątek I.                        3     3      100.000000
Thompson J.                       1     1      100.000000
Volynets K.                       1     1      100.000000
Michelsen A.                      1     1      100.000000
Kerber A.                         1     1      100.000000
Kalinina A.                       1     1      100.000000
Humbert U.                        4     4      100.000000
Haddad Maia B.                    1     1      100.000000
Tsurenko L.                       1     1      100.000000
Paolini J.                        1     1      100.000000
Nardi L.                          2     2      100.000000
Vekić D.      